In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
# Plain LLM

def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [3]:
# testing the black box, where text comes in and text comes out

llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s going on?'

In [3]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes, probably — but it depends on the course’s enrollment rules and whether it’s still open.

If you want, I can help you figure out the best next step:
- check whether late enrollment is allowed,
- draft a message to the instructor/admin,
- or help you ask in a polite way.

A simple message could be:

> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll at this point? Thank you.

If you tell me the course name or platform, I can help you tailor the message.


In [ ]:
'''
The LLM gives a generic answer. It might say "you can usually join" or "check the course website." 
It doesn't know about our specific Zoomcamp courses, their enrollment policies, or their schedules. 
It tries to be helpful, but has no idea about actual enrollment status or policies.

This is different from a question like "how do I cook salmon?" - the LLM knows the answer because cooking salmon is common knowledge. 
But our courses are not in the training data.
'''

In [4]:
# Adding some context manually:

context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [5]:
# now buiding a prompt that takes into account the context

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [ ]:
"""
RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. 
We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. 
That search step is what gives the LLM the context it needs to answer correctly.

What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. 
What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.
"""

In [6]:
# putting this idea in code, it will look something like:

def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
"""
The LLM only sees the documents we hand it, so its answers are grounded in our data. If the right document is retrieved, the answer is good. 
If it's not, the LLM gets the wrong context and the answer is wrong. Your model is only as good as your retrieval, 
so search quality matters a lot for RAG.
"""

In [7]:
# The dataset for this exercise

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

#This returns a list of courses. Each course has a path field that points to its FAQ data.

In [8]:
# fecthing all FAQ documents from all courses:

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [12]:
# looking at one entry:

documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [ ]:
# searching engine:

score = sim(query, document)

In [ ]:
"""
For each document in the database, you compute this score. Then you rank all documents by score and return the top N. 
What makes a search engine different from another search engine is what sim actually computes.

* text/lexical search (covered in this section): sim counts how many words the query and the document share. 
It looks at the surface form, the actual words, and matches them exactly.

* vector/semantic search (covered in module 2): sim compares the meaning of the query and the document. Same function, different similarity measure.
"""

In [ ]:
"""
Searching matters because we have around 1100 documents. Sending all of them to the LLM would be expensive and slow. 
The model would get confused with too much data. Search finds the most relevant documents to send instead.
"""

In [9]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [10]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

"""
We used boost_dict to give the question field more weight (2.0 instead of the default 1.0) and section less weight (0.5). 
Query words appearing in the question field is a stronger signal than them appearing in the section name.
We used filter_dict to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.
"""

"\nWe used boost_dict to give the question field more weight (2.0 instead of the default 1.0) and section less weight (0.5). \nQuery words appearing in the question field is a stronger signal than them appearing in the section name.\nWe used filter_dict to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.\n"

In [8]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [11]:
# minsearch supports field boosting to reflect this fact that the question fields weight more than section fields:

results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

# This is the same boosting mechanism used by Elasticsearch and Lucene.

In [12]:
# Sometimes you want to restrict the search to a specific course. minsearch supports keyword filtering:

results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [13]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [14]:
# Wrapping it in a function:

def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(question)

TypeError: 'list' object is not callable

In [15]:
# Instructions (also called the system prompt): this tells the LLM how to behave. It never changes, so it's the same for every request.
# The instructions tell the LLM its role and how to answer:

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [16]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [17]:
# Building the context:

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [18]:
# Building the prompt:

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [19]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [20]:
# LLM

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [21]:
# the response:

response.output[0]

ResponseOutputMessage(id='msg_06714ed9a1374605006a399033cd2481a291d08f4e272c8fe6', content=[ResponseOutputText(annotations=[], text='Yes — you can join now.\n\nYou can start learning and submitting homework as long as the submission form is still open. If you want to receive a certificate, make sure you submit your project while the course is still accepting submissions.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [22]:
response.output[0].content[0].text

'Yes — you can join now.\n\nYou can start learning and submitting homework as long as the submission form is still open. If you want to receive a certificate, make sure you submit your project while the course is still accepting submissions.'

In [23]:
response.output_text

'Yes — you can join now.\n\nYou can start learning and submitting homework as long as the submission form is still open. If you want to receive a certificate, make sure you submit your project while the course is still accepting submissions.'

In [24]:
response.usage

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=50, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=530)

In [25]:
# Calculating the price:

"""
For model gpt-5.4-mini:
Input: $0.75 per million tokens
Output: $4.50 per million tokens
"""

input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000585

In [26]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

"""
OpenAI accepts both developer and system for the instruction role. 
There may be some difference between them, but in practice I don't see it change the result either way. We use developer in this course.
"""

In [27]:
# Updated LLM function:

def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [28]:
# Full RAG:

def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [29]:
# testing:

answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want a certificate, you need to submit your project while submissions are still open.


In [30]:
rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a **live cohort** and **pass the Capstone project**.\n\nA few important notes:\n- **Self-paced mode does not include a certificate.**\n- **Homework is not mandatory** for the certificate.\n- You must also **peer-review 3 capstones** during the live course period.\n\nIf you want, I can also tell you how to make sure your name appears correctly on the certificate.'

In [1]:
# crerating ingest.py

import requests
from minsearch import Index

def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = "https://datatalks.club/faq"

    for course in courses_raw:
        course_url = f"""{url_prefix}{course["path"]}"""
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents

def build_index(documents):
    index = Index(
        text_fields=["question", "section", "answer"],
        keyword_fields=["course"]
    )
    index.fit(documents)
    return index

In [7]:
# creating rag.py

INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'question': 3.0, 'section': 0.5}
        filter_dict = {'course': self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['section'])
            lines.append('Q: ' + doc['question'])
            lines.append('A: ' + doc['answer'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer


In [10]:
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-23 17:10:05--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0.002s  

2026-06-23 17:10:05 (1013 KB/s) - ‘rag_helper.py.1’ saved [2134/2134]

--2026-06-23 17:10:05--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 20

In [11]:
# testing:

from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now.

If you want a certificate, make sure to submit your project while submissions are still open.


In [12]:
# testing override instructions

custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [13]:
# trying more questions:

assistant.rag("How do I get a certificate?")
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join after the course has started.\n\nIf you want a certificate, you need to submit your project while the course is still accepting submissions, and certificates are only awarded for the live cohort, not self-paced mode.'

In [2]:
# RAG with sqlitesearch

from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv
from sqlitesearch import TextSearchIndex

openai_client = OpenAI()
load_dotenv()

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [18]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [ ]:
# Summary:

"""
minsearch: single process, in-memory only, re-indexes on every startup. Use when data is small and indexing is fast.
sqlitesearch: separate ingestion and query, file-based (SQLite), opens existing index. Use when data is large or ingestion is slow.
"""

In [19]:
# Closing the database connection:

sqlite_index.close()

In [3]:
# Setting up RAG

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# loading data and building the search index:

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

# Creating the assistant:

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [23]:
# testing the assistant:

assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. **Install Ollama** from [https://ollama.com/download](https://ollama.com/download) for your operating system.\n   - **macOS**: download the `.pkg` and install it.\n   - **Windows**: download the `.msi` and install it.\n   - **Linux**: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally** by opening a terminal and running:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. **Optional: test the local server** with:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [24]:
# forcing a typo:

assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about **running Ollama locally**.\n\nThe closest related note says you can **run the course locally instead of Codespaces** if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module, and to **document your setup** so it stays reproducible.\n\nIf you meant **“Can I run the course locally?”**, the answer is yes.'

In [4]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [ ]:
"""
The LLM is a passenger here, not a driver. It never sees the bad search results, so it can't try again with a corrected query.
"""

In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Usually, yes — if the course is still open for enrollment and you meet any prerequisites.\n\nTo help you accurately, I need a bit more info:\n- Which course are you referring to?\n- Is it already in progress or starting soon?\n- Do you want help finding the enrollment link or checking eligibility?\n\nIf you want, send me the course name and I’ll help you figure out the next step.'

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll can I join"}', call_id='call_NWLqTpg7G4gou82yBCUTGyBe', name='search', type='function_call', id='fc_0cfdd3bda5bb4b54006a3c182ad178819f87e9a7798aab4eb9', namespace=None, status='completed')]

In [9]:
# Executing the function and sending back the result:

# The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [11]:
# Now we send this result back to the model. First, we add the model's output to the 
# conversation history - the model needs to see its own function call. Then we add the tool result.

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

# The call_id links the tool result to the specific function call the model requested. 
# If the model makes multiple function calls in one turn, each one gets its own call_id.

In [12]:
# Calling the API again, now with the expanded history:

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

BadRequestError: Error code: 400 - {'error': {'message': 'Duplicate item found with id fc_0cfdd3bda5bb4b54006a3c182ad178819f87e9a7798aab4eb9. Remove duplicate items from your input and try again.', 'type': 'invalid_request_error', 'param': 'input', 'code': None}}

In [13]:
# Token usage and cost:

usage = response.usage
usage.input_tokens, usage.output_tokens

(69, 24)

In [14]:
"""
For each model the provider publishes a price per million input tokens and per million output tokens. 
Plug those numbers in to convert tokens to dollars.
"""

def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
